# 多市场与逐笔回放

按顺序运行。示例使用合成行情、零交易费和示例税率，不能用作真实市场历史结论。真实资料格式见 docs/多市场与真实权益.md。

In [ ]:
from pathlib import Path
import sys, json, tempfile
import numpy as np
root = Path.cwd()
if not (root / "qbt").exists(): root = root.parent
sys.path.insert(0, str(root))
import qbt
from examples.market_switch import fixture

## 1. 在下面选 CN、US 或 KR
更换市场后重新运行本格及后续格。真实使用时将 fixture 换为 load_market_csv，并加载自己的 ReferenceData。

In [ ]:
MARKET = "US"
d, manifest = fixture(MARKET)
out_base = root / "reports" / "market_upgrade" / "notebook"
out_base.mkdir(parents=True, exist_ok=True)
out = Path(tempfile.mkdtemp(prefix="run_", dir=out_base))
ref_file = out / "reference.json"
ref_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ref = qbt.ReferenceData.from_json(ref_file)
print(MARKET, qbt.market_profile(MARKET).currency)

## 2. 在这一格粘贴或编写策略代码
保留函数名 strategy。下面是最小买入持有示例；可以使用 ctx.ret、ctx.universe、ctx.long 等接口。

In [ ]:
def strategy(ctx):
    return ctx.long(np.ones(ctx.N, dtype=bool), stops={"hard": 0.05})

## 3. 执行并查看报告

In [ ]:
r = qbt.run(d, strategy, market=MARKET, reference_data=ref,
            capital=10000 * (7000 if MARKET == "KR" else 1),
            rebalance_tol=1, max_adv_pct=1)
print(r)
print("币种", r.currency, "权益税款", r.account.taxes)
report = qbt.report(r, path=str(out / "report.html"), plotly=False)
print(report.path)

## 4. 直接订单入口
订单策略可单独编写。不要把目标持仓策略传给 run_orders。

In [ ]:
def order_strategy(ctx):
    if ctx.T == 1:
        ctx.order(d.codes[0], 100 if MARKET == "CN" else 1, tif="GTC")
order_result = qbt.run_orders(d, order_strategy, market=MARKET, reference_data=ref, capital=1000000, max_adv_pct=1)
print([(f.code, f.quantity, f.price) for f in order_result.order_fills])

## 5. 逐笔队列示例
此格独立使用美股单会话的合成 L3 消息。自己的买单在 ahead 后、behind 前入队；ahead 撤掉后，behind 的 40 股成交证据才分给自己的订单。不是从日线推测队列。

In [ ]:
us_d, us_manifest = fixture("US")
us_calendar = qbt.SessionCalendar(us_manifest["sessions"], "US", "synthetic")
feed = [
 qbt.L3Event("2024-03-08T09:30:00",1,"add","DEMO","ahead","buy",10.,100),
 qbt.L3Event("2024-03-08T09:30:01",2,"add","DEMO","behind","buy",10.,100),
 qbt.L3Event("2024-03-08T09:30:02",3,"cancel","DEMO","ahead","buy",10.,100),
 qbt.L3Event("2024-03-08T09:30:03",4,"execute","DEMO","behind","buy",10.,40)]
submitted = []
def maker(ctx):
    if not submitted:
        submitted.append(ctx.order(qbt.OrderRequest("DEMO",100,limit_price=10.)))
l3 = qbt.run_l3(feed, maker, profile="US", calendar=us_calendar, codes=["DEMO"],
                capital=10000, costs=qbt.Costs(0,0,0,stamp_duty=0))
print(l3.fills)
print(l3.save_json(out / "l3.json"))

真实使用：`d = qbt.load_market_csv("bars.csv", MARKET)`，`ref = qbt.ReferenceData.from_json("reference.json")`。真实美股、韩股行情和逐笔队列资料需自行提供；不能把这份合成样例或 A 股矩阵改个市场名当作真实数据。